In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import statsmodels.api as sm
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import logit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pandas as pd

import os
import sys
utils_path = os.path.abspath(
    os.path.join(os.getcwd(), '..', '2_data_analysis', 'utils')
)

sys.path.append(utils_path)

import plot_style


# ── Palette and style ────────────────────────────────────────────────────────
C_AGR   = '#C0392B'   # steel blue  — with acuerdo
C_NOAGR = '#2C6E8A'   # muted red   — without acuerdo

## **Conflict Panel**

---

1. Keep all the conflicts that sign agreements in the sample.

In [2]:
df_panel = pd.read_csv('../../data/output/conflict_level/conflict_panel.csv', low_memory=False)

#df_panel_quarterly = pd.read_csv('../../data/output/conflict_level/conflict_panel_quarters.csv')
#select the treated conflicts that were defines for csdid
#active_conflicts = df_panel_quarterly[df_panel_quarterly['first_agreement']==1]['conflict_id'].unique()

#replace the column ever_agreement in conflict_level with 1 only for those conflicts that are in active_conflicts
#df_panel['ever_agreement'] = df_panel.apply(lambda row: 1 if row['conflict_id'] in active_conflicts else 0, axis=1)

#create ever agreement column in df_panel that is 1 if the conflict has ever signed an agreement and 0 otherwise
df_panel['ever_agreement'] = df_panel.groupby('conflict_id')['agreement'].transform('max')

print(f"Total conflicts: {len(df_panel['conflict_id'].unique())}")
print(f"With agreement:   {df_panel.loc[df_panel['ever_agreement']==1]['conflict_id'].nunique()}")
print(f"Without agreement: {df_panel.loc[df_panel['ever_agreement']==0]['conflict_id'].nunique()}")


Total conflicts: 201
With agreement:   104
Without agreement: 97


In [3]:
# En conflict_panel.ipynb, verificar cuántos conflicts tienen
# períodos de dormancia sustanciales (> 4 trimestres con best=0 consecutivos)

df_panel['dormant'] = (df_panel['best'] == 0).astype(int)

max_consecutive_dormancy = (
    df_panel
    .sort_values(['conflict_id', 'year_mo'])
    .groupby('conflict_id')['dormant']
    .apply(lambda x: (x * (x.groupby((x != x.shift()).cumsum()).cumcount() + 1)).max())
)

print("Distribución de la dormancia máxima consecutiva (meses):")
print(max_consecutive_dormancy.describe())
print(f"\nConflictos con > 4 trimestres (12 meses) de dormancia: "
      f"{(max_consecutive_dormancy > 12).sum()} de {len(max_consecutive_dormancy)}")


Distribución de la dormancia máxima consecutiva (meses):
count    201.000000
mean     248.393035
std      123.865813
min        2.000000
25%      147.000000
50%      290.000000
75%      342.000000
max      431.000000
Name: dormant, dtype: float64

Conflictos con > 4 trimestres (12 meses) de dormancia: 196 de 201


## **UCDP - Incompatibility Type**

---

In [12]:
#merge the incompatibility column from UCDP/PRIO
df_prio = pd.read_csv('../../data/input/ucdp/UcdpPrioConflict_v25_1.csv', low_memory=False)
df_panel = df_panel.merge(df_prio[['conflict_id', 'year', 'incompatibility', 'type_of_conflict']], left_on = ['conflict_id', 'year'], right_on = ['conflict_id', 'year'], how = 'left')
df_panel['incompatibility'] = df_panel.groupby('conflict_id')['incompatibility'].ffill().bfill().astype(int)
type_of_conflict_map = {1: "extrasystemic", 2: "intrastate", 3: "interstate", 4: "internationalized_intrastate"}
df_panel['type_of_conflict'] = df_panel.groupby('conflict_id')['type_of_conflict'].ffill().bfill().map(type_of_conflict_map)
df_panel[['conflict_id', 'year_mo', 'incompatibility', 'type_of_conflict']]

,conflict_id,year_mo,incompatibility,type_of_conflict
0,205,1990-04,1,interstate
1,205,1990-05,1,interstate
2,205,1990-06,1,interstate
3,205,1990-07,1,interstate
4,205,1990-08,1,interstate
...,...,...,...,...
34989,16292,2019-08,1,interstate
34990,16292,2019-09,1,interstate
34991,16292,2019-10,1,interstate
34992,16292,2019-11,1,interstate


## **UCDP - Number of Factions**

---

- Is there a way to add the ethnic group of the factions involved?

In [13]:
from functions.build_number_factions import (
    build_faction_variables
)

df_panel = build_faction_variables(
    df_panel,
    "../../data/input/ucdp/Dyadic_v25_1.csv")

df_panel[['conflict_id', 'year_mo', 'n_factions_combined', 'new_rebel_entry']]

,conflict_id,year_mo,n_factions_combined,new_rebel_entry
0,205,1990-04,1.0,0
1,205,1990-05,1.0,0
2,205,1990-06,1.0,0
3,205,1990-07,1.0,0
4,205,1990-08,1.0,0
...,...,...,...,...
34989,16292,2019-08,1.0,0
34990,16292,2019-09,1.0,0
34991,16292,2019-10,1.0,0
34992,16292,2019-11,1.0,0


## **UCDP - Termination Conflict Type**

---

In [14]:
from functions.build_conflict_termination import (
    build_conflict_termination
)

df_panel = build_conflict_termination(
    df_panel,
    "../../data/input/ucdp/UCDPConflictTerminationDataset_v4_2024_Conflict.csv"
)

df_panel[['conflict_id', 'year_mo', 'termination_outcome_label', 'termination_outcome_group']]

,conflict_id,year_mo,termination_outcome_label,termination_outcome_group
0,205,1990-04,Low activity,Low activity
1,205,1990-05,Low activity,Low activity
2,205,1990-06,Low activity,Low activity
3,205,1990-07,Low activity,Low activity
4,205,1990-08,Low activity,Low activity
...,...,...,...,...
34989,16292,2019-08,Low activity,Low activity
34990,16292,2019-09,Low activity,Low activity
34991,16292,2019-10,Low activity,Low activity
34992,16292,2019-11,Low activity,Low activity


## **UCDP - Experience Fighting**
---

In [15]:
# from functions.build_experience import (
#     build_experience
# )
from functions.build_experience_compound import build_experience_expanded


# df_panel = build_experience(
#     df_panel,
#     "../../data/input/ucdp/Dyadic_v25_1.csv",
#     delta=0.95
# )

df_panel = build_experience_expanded(
    df_panel,
    "../../data/input/ucdp/Dyadic_v25_1.csv",
    delta=0.95,
    weight_direct=1.0,
    weight_government=0.5,
)

df_panel[['conflict_id', 'year_mo', 'experience_total', 'exp_direct', 'exp_government']]

Experience computed for 201 conflicts:
  exp_direct = 0 in 133/201 conflicts (66%)
  experience_total = 0 in 77/201 conflicts (38%)
  (fewer zeros = more variation for IA proxy)


,conflict_id,year_mo,experience_total,exp_direct,exp_government
0,205,1990-04,12.164151,8.270352,7.787599
1,205,1990-05,12.164151,8.270352,7.787599
2,205,1990-06,12.164151,8.270352,7.787599
3,205,1990-07,12.164151,8.270352,7.787599
4,205,1990-08,12.164151,8.270352,7.787599
...,...,...,...,...,...
34989,16292,2019-08,1.960904,0.000000,3.921809
34990,16292,2019-09,1.960904,0.000000,3.921809
34991,16292,2019-10,1.960904,0.000000,3.921809
34992,16292,2019-11,1.960904,0.000000,3.921809


## **Vdem Variables**

---

In [16]:
from functions.build_vdem_variables import (
    build_vdem_dataset,
    merge_vdem_to_panel
)

# Build VDEM
df_vdem = build_vdem_dataset(
    "../../data/input/v_dem/V-Dem-CY-Full+Others-v16.csv"
)

#rolling window columns 
VDEM_ROLLING_VARS = [
    "vdem_state_territorial_control",
    "vdem_horizontal_accountability",
    "vdem_judicial_constraints",
    "vdem_legislative_constraints",
    "vdem_neopatrimonial",
    "vdem_clientelism",
    "vdem_predictable_enforcement",
    "vdem_wb_political_stability",
    "vdem_wb_government_effectiveness",
    "vdem_public_adm_capacity",

]
VDEM_ROLLING_VARS_mean_lag5 = [var + "_pre5y" for var in VDEM_ROLLING_VARS]


# variables to merge
VDEM_columns = ['cow_code','year','vdem_state_territorial_control','vdem_neopatrimonial','vdem_horizontal_accountability',
        'vdem_judicial_constraints','vdem_legislative_constraints', 'vdem_regime_type', "vdem_predictable_enforcement", 
         "vdem_wb_political_stability",  "vdem_wb_government_effectiveness", "vdem_public_adm_capacity"]

# Merge into panel
df_panel = merge_vdem_to_panel(
    df_panel,
    df_vdem,
    VDEM_columns + VDEM_ROLLING_VARS_mean_lag5,
    "../../data/input/ucdp/GEDEvent_v25_1.csv"
)

df_panel[['conflict_id', 'year_mo']+VDEM_columns+VDEM_ROLLING_VARS_mean_lag5]

/Users/luciasauer/Library/CloudStorage/GoogleDrive-lucia.sauer@bse.eu/Mi unidad/EconAI/1_agreements_violence/agreements_violence/src/5_survival_analysis/functions/build_vdem_variables.py:203: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,conflict_id,year_mo,cow_code,year,vdem_state_territorial_control,vdem_neopatrimonial,vdem_horizontal_accountability,vdem_judicial_constraints,vdem_legislative_constraints,vdem_regime_type,...,vdem_state_territorial_control_pre5y,vdem_horizontal_accountability_pre5y,vdem_judicial_constraints_pre5y,vdem_legislative_constraints_pre5y,vdem_neopatrimonial_pre5y,vdem_clientelism_pre5y,vdem_predictable_enforcement_pre5y,vdem_wb_political_stability_pre5y,vdem_wb_government_effectiveness_pre5y,vdem_public_adm_capacity_pre5y
0,205,1990-04,630.0,1990,91.667,0.748,0.176,0.146,0.627,1.0,...,92.2668,0.1668,0.1460,0.6012,0.7438,0.3542,-0.2816,NaN,NaN,-1.0800
1,205,1990-05,630.0,1990,91.667,0.748,0.176,0.146,0.627,1.0,...,92.2668,0.1668,0.1460,0.6012,0.7438,0.3542,-0.2816,NaN,NaN,-1.0800
2,205,1990-06,630.0,1990,91.667,0.748,0.176,0.146,0.627,1.0,...,92.2668,0.1668,0.1460,0.6012,0.7438,0.3542,-0.2816,NaN,NaN,-1.0800
3,205,1990-07,630.0,1990,91.667,0.748,0.176,0.146,0.627,1.0,...,92.2668,0.1668,0.1460,0.6012,0.7438,0.3542,-0.2816,NaN,NaN,-1.0800
4,205,1990-08,630.0,1990,91.667,0.748,0.176,0.146,0.627,1.0,...,92.2668,0.1668,0.1460,0.6012,0.7438,0.3542,-0.2816,NaN,NaN,-1.0800
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34989,16292,2019-08,702.0,2019,88.500,0.970,-1.538,0.034,0.044,1.0,...,88.7306,-1.5234,0.0354,0.0472,0.9706,0.7832,-1.9722,-0.6482,-0.736,-1.3524
34990,16292,2019-09,702.0,2019,88.500,0.970,-1.538,0.034,0.044,1.0,...,88.7306,-1.5234,0.0354,0.0472,0.9706,0.7832,-1.9722,-0.6482,-0.736,-1.3524
34991,16292,2019-10,702.0,2019,88.500,0.970,-1.538,0.034,0.044,1.0,...,88.7306,-1.5234,0.0354,0.0472,0.9706,0.7832,-1.9722,-0.6482,-0.736,-1.3524
34992,16292,2019-11,702.0,2019,88.500,0.970,-1.538,0.034,0.044,1.0,...,88.7306,-1.5234,0.0354,0.0472,0.9706,0.7832,-1.9722,-0.6482,-0.736,-1.3524


In [17]:
# ── V-Dem pre5y: two versions for survival analysis ───────────────────────────
#
# Version 1 — time-invariant (_pre5y_start):
#   The 5-year rolling mean is frozen at the conflict's UCDP start_date year.
#   One value per conflict; captures institutional quality at onset.
#
# Version 2 — time-varying (_pre5y):
#   The rolling mean already in df_panel updates year-by-year during the conflict.
#   For treated conflicts it will evolve up to the agreement quarter;
#   for right-censored conflicts it evolves over the full observed spell.
#   No extra code needed — these columns will pass through _AGG below.

VDEM_PRE5Y_VARS = [
    "vdem_state_territorial_control_pre5y",
    "vdem_horizontal_accountability_pre5y",
    "vdem_judicial_constraints_pre5y",
    "vdem_legislative_constraints_pre5y",
    "vdem_neopatrimonial_pre5y",
    "vdem_wb_political_stability_pre5y",
]

# conflict start year from UCDP start_date
conflict_start_yr = (
    df_panel.groupby('conflict_id')['start_date']
    .first()
    .reset_index()
    .assign(start_year=lambda d: pd.to_datetime(d['start_date']).dt.year)
    [['conflict_id', 'start_year']]
)

# look up the _pre5y value at the start year for each conflict
_start_pre5y = (
    df_panel[['conflict_id', 'year'] + VDEM_PRE5Y_VARS]
    .merge(conflict_start_yr, on='conflict_id')
    .query('year == start_year')
    .drop_duplicates('conflict_id')
    .rename(columns={v: v + '_start' for v in VDEM_PRE5Y_VARS})
    [['conflict_id'] + [v + '_start' for v in VDEM_PRE5Y_VARS]]
)

df_panel = df_panel.merge(_start_pre5y, on='conflict_id', how='left')

VDEM_PRE5Y_START_VARS = [v + '_start' for v in VDEM_PRE5Y_VARS]
print("pre5y_start (time-invariant) — NaN counts:")
print(df_panel[VDEM_PRE5Y_START_VARS].isna().sum())
print(f"\nUnique values per conflict (should be 1):")
print(df_panel.groupby('conflict_id')[VDEM_PRE5Y_START_VARS[:3]].nunique().max())

pre5y_start (time-invariant) — NaN counts:
vdem_state_territorial_control_pre5y_start        0
vdem_horizontal_accountability_pre5y_start        0
vdem_judicial_constraints_pre5y_start             0
vdem_legislative_constraints_pre5y_start        796
vdem_neopatrimonial_pre5y_start                   0
vdem_wb_political_stability_pre5y_start       26127
dtype: int64

Unique values per conflict (should be 1):
vdem_state_territorial_control_pre5y_start    1
vdem_horizontal_accountability_pre5y_start    1
vdem_judicial_constraints_pre5y_start         1
dtype: int64


In [18]:
# ── Impute missing pre5y_start values ─────────────────────────────────────────
# vdem_legislative_constraints_pre5y_start: ~4 conflicts missing → regional mean
# vdem_wb_political_stability_pre5y_start:  ~111 conflicts missing (WB data only
#   available from 1996; conflicts starting before ~2001 have no 5y pre-mean).
#   → regional mean imputation for individual specs; excluded from PCA.
#
# Strategy: for each missing conflict, impute using the mean of conflicts in the
# same region whose start year is within ±5 years.

# region and start year are constant per conflict
conflict_meta = (
    df_panel.drop_duplicates('conflict_id')[['conflict_id', 'region', 'start_date']]
    .assign(start_year=lambda d: pd.to_datetime(d['start_date']).dt.year)
    .merge(_start_pre5y, on='conflict_id', how='left')   # bring in the start values
)

IMPUTE_VARS = [
    'vdem_legislative_constraints_pre5y_start',
    'vdem_wb_political_stability_pre5y_start',
]

for var in IMPUTE_VARS:
    missing_mask = conflict_meta[var].isna()
    n_missing = missing_mask.sum()
    if n_missing == 0:
        continue
    for idx in conflict_meta.index[missing_mask]:
        row = conflict_meta.loc[idx]
        # same region, start year within ±5 years, non-missing
        candidates = conflict_meta[
            (conflict_meta['region'] == row['region']) &
            (conflict_meta['start_year'].between(row['start_year'] - 5,
                                                  row['start_year'] + 5)) &
            conflict_meta[var].notna()
        ]
        if len(candidates) > 0:
            conflict_meta.loc[idx, var] = candidates[var].mean()
        else:
            # fallback: global mean of same start-year decade
            decade = (row['start_year'] // 10) * 10
            candidates_global = conflict_meta[
                conflict_meta['start_year'].between(decade, decade + 9) &
                conflict_meta[var].notna()
            ]
            if len(candidates_global) > 0:
                conflict_meta.loc[idx, var] = candidates_global[var].mean()
    print(f"{var}: {n_missing} imputed → "
          f"{conflict_meta[var].isna().sum()} remaining NaN")

# merge imputed values back into df_panel (drop old cols first to avoid duplicates)
df_panel = df_panel.drop(columns=IMPUTE_VARS, errors='ignore')
df_panel = df_panel.merge(
    conflict_meta[['conflict_id'] + IMPUTE_VARS],
    on='conflict_id', how='left'
)

print("\npost-imputation NaN counts in df_panel:")
print(df_panel[VDEM_PRE5Y_START_VARS].isna().sum())

vdem_legislative_constraints_pre5y_start: 4 imputed → 0 remaining NaN
vdem_wb_political_stability_pre5y_start: 111 imputed → 4 remaining NaN

post-imputation NaN counts in df_panel:
vdem_state_territorial_control_pre5y_start       0
vdem_horizontal_accountability_pre5y_start       0
vdem_judicial_constraints_pre5y_start            0
vdem_legislative_constraints_pre5y_start         0
vdem_neopatrimonial_pre5y_start                  0
vdem_wb_political_stability_pre5y_start       1284
dtype: int64


## **Similarity Vote - UN Voting**
---

Create variables that measure country alignment with the Security Council (SC) members, looking at the voting similarity in the UN General Assembly.

- <code>influence</code>: sum of same votes with SC members per year
- <code>influence_veto</code> sum of same votes with SC members per year weighted by veto power (10 for P5, 1 for non-P5)
- <code>influence_gdp</code>: sum of same votes with SC members per year weighted by GDP
- <code>influence_gdp</code>: sum of same votes with SC members per year weighted by GDP in log form (to account for skewness)

>The construction of the similarity scores dataset will be ran once, because it takes time.
>The resulting dataset will be saved in data/output/country_level/similarity_scores.csv, and just imported directly and then merged with the main panel dataset.

In [19]:
# from functions.build_sim_voting import build_influence_scores
# similarity_scores = build_influence_scores(
#     votes_path = '../../data/input/un/2025_9_19_ga_voting.csv',
#     members_sc_path = '../../data/input/un/DPPA-SCMEMBERSHIP.csv',
#     isocodes_path = '../../data/input/isocodes/isocodes_appended.csv',
#     gdp_path = '../../data/input/gdp_pc/gdp_pc.csv',
#     )
# similarity_scores.to_csv('../../data/output/country_level/similarity_voting_scores.csv', index=False)
similarity_scores = pd.read_csv('../../data/output/country_level/similarity_voting_scores.csv')
#import similarity_voting_scores and merge with panel
df_panel = df_panel.merge(
    similarity_scores[['isocode', 'year', 'influence', 'influence_veto', 'influence_gdp', 'influence_log_gdp']],
    left_on = ['isocode', 'year'],
    right_on = ['isocode', 'year'],
    how = 'left'
)
df_panel[['conflict_id','isocode', 'year_mo', 'influence_veto','influence', 'influence_gdp', 'influence_log_gdp','best']]

,conflict_id,isocode,year_mo,influence_veto,influence,influence_gdp,influence_log_gdp,best
0,205,IRN,1990-04,33.134376,10.706259,0.150876,8.651369,0.0
1,205,IRN,1990-05,33.134376,10.706259,0.150876,8.651369,1.0
2,205,IRN,1990-06,33.134376,10.706259,0.150876,8.651369,23.0
3,205,IRN,1990-07,33.134376,10.706259,0.150876,8.651369,7.0
4,205,IRN,1990-08,33.134376,10.706259,0.150876,8.651369,0.0
...,...,...,...,...,...,...,...,...
34989,16292,TJK,2019-08,35.287722,10.426399,0.272676,8.742290,0.0
34990,16292,TJK,2019-09,35.287722,10.426399,0.272676,8.742290,0.0
34991,16292,TJK,2019-10,35.287722,10.426399,0.272676,8.742290,0.0
34992,16292,TJK,2019-11,35.287722,10.426399,0.272676,8.742290,17.0


## **UCDP External Support — Rebel External Support**

---

Binary indicator: rebel faction has foreign military patron in that conflict-year.
Source: UCDP External Support Dataset (ESD) triad-year, v18.1 (covers 1975–2017).
Conflicts after 2017 imputed as 0 (conservative; flag in robustness checks).

In [20]:
_esd = pd.read_excel(
    "../../data/input/external_support/ucdp-esd-ty-181.xlsx",
    usecols=["conflict_id", "year", "actor_nonstate", "ext_sum"]
)

# rebel faction (actor_nonstate==1) with any external support (ext_sum>0)
_rebel_ext = (
    _esd[(_esd["actor_nonstate"] == 1) & (_esd["ext_sum"] > 0)]
    .groupby(["conflict_id", "year"])
    .size()
    .reset_index(name="_n")
    .assign(rebel_ext_support=1)[["conflict_id", "year", "rebel_ext_support"]]
    .drop_duplicates()
)

df_panel = df_panel.merge(_rebel_ext, on=["conflict_id", "year"], how="left")
df_panel["rebel_ext_support"] = df_panel["rebel_ext_support"].fillna(0).astype(int)

print(f"rebel_ext_support = 1: {df_panel['rebel_ext_support'].mean():.1%} of conflict-months")
print(f"(ESD covers 1975–2017; conflicts/years outside that window imputed as 0)")

rebel_ext_support = 1: 28.0% of conflict-months
(ESD covers 1975–2017; conflicts/years outside that window imputed as 0)


## **Spell Dataset**

---

A **survival dataset** (or spell dataset) records each conflict's history as a sequence of periods until an event occurs or the observation ends without one. Each row is one **conflict × period** observation; the event of interest is the signing of the **first peace agreement**.

Two filtering steps define the spell:

1. **Trim to the active conflict window** — keep only observations between `start_date` and `end_date` (UCDP conflict onset and last recorded activity).
2. **Censor at first agreement** — drop all observations after the first agreement. Each conflict's spell ends either at the event quarter (`is_first_agreement = 1`) or at the last observed period (right-censored, `ever_agreement = 0`).

This structure supports a **discrete-time hazard model**: at each period, the conflict either transitions to an agreement or remains at risk. The primary estimation dataset is aggregated at the **quarterly level** (`spell_q`), which is the unit of analysis for the ClogLog model.

In [21]:
# En conflict_panel.ipynb, verificar cuántos conflicts tienen
# períodos de dormancia sustanciales (> 4 trimestres con best=0 consecutivos)

df_panel = df_panel[(df_panel['year_mo']>=df_panel['start_date']) & (df_panel['year_mo']<=df_panel['end_date'])]
df_panel = df_panel[((df_panel["first_agreement_date"] == 0) | (df_panel["year_mo_numeric"]<=df_panel["first_agreement_date"]))].copy()
df_panel = df_panel.sort_values(["conflict_id", "year_mo_numeric"])

df_panel['dormant'] = (df_panel['best'] == 0).astype(int)

max_consecutive_dormancy = (
    df_panel
    .sort_values(['conflict_id', 'year_mo'])
    .groupby('conflict_id')['dormant']
    .apply(lambda x: (x * (x.groupby((x != x.shift()).cumsum()).cumcount() + 1)).max())
)

print("Distribución de la dormancia máxima consecutiva (meses):")
print(max_consecutive_dormancy.describe())
print(f"\nConflictos con > 4 trimestres (12 meses) de dormancia: "
      f"{(max_consecutive_dormancy > 12).sum()} de {len(max_consecutive_dormancy)}")


Distribución de la dormancia máxima consecutiva (meses):
count    201.000000
mean      25.258706
std       50.570374
min        0.000000
25%        1.000000
50%        7.000000
75%       20.000000
max      352.000000
Name: dormant, dtype: float64

Conflictos con > 4 trimestres (12 meses) de dormancia: 73 de 201


In [22]:
df_panel_capped = df_panel[(df_panel['year_mo']>=df_panel['start_date']) & (df_panel['year_mo']<=df_panel['end_date'])]
pre_treatment = df_panel_capped[((df_panel_capped["first_agreement_date"] == 0) | (df_panel_capped["year_mo_numeric"]<=df_panel_capped["first_agreement_date"]))].copy()
pre_treatment = pre_treatment.sort_values(["conflict_id", "year_mo_numeric"])
pre_treatment[['conflict_id', 'year_mo', 'first_agreement_date','year_mo_numeric','ever_agreement']]


,conflict_id,year_mo,first_agreement_date,year_mo_numeric,ever_agreement
0,205,1990-04,0,16,0
1,205,1990-05,0,17,0
2,205,1990-06,0,18,0
3,205,1990-07,0,19,0
4,205,1990-08,0,20,0
...,...,...,...,...,...
34989,16292,2019-08,0,368,0
34990,16292,2019-09,0,369,0
34991,16292,2019-10,0,370,0
34992,16292,2019-11,0,371,0


In [23]:
# ── 1. Event: is first agreement in this month? ────────────────────────────────
pre_treatment['is_first_agreement'] = (
    (pre_treatment['ever_agreement'] == 1) &
    (pre_treatment['year_mo_numeric'] == pre_treatment['first_agreement_date'])
).astype(int)
print(pre_treatment.groupby('conflict_id')['is_first_agreement'].sum().value_counts())


is_first_agreement
1    103
0     98
Name: count, dtype: int64


In [24]:
# ── 2. BASELINE HAZARD: log of conflict age ─────────────────────────
pre_treatment['log_conflict_age'] = np.log1p(pre_treatment['conflict_age'])

# ── 3. DESTRUCTION: cumulative fatalities until previous month ──────────────
pre_treatment['log_cum_deaths'] = np.log1p(
    pre_treatment.groupby('conflict_id')['best']
    .transform(lambda x: x.cumsum().shift(1))
    .fillna(0)
)

In [25]:
# ── 4. COMMIT Proxy ─────────────────────────
#count the number of conflicts per country and then append to conflict_level
pre_treatment['multiple_conflicts'] = pre_treatment.groupby(['country', 'year_mo'])['conflict_id'].transform('nunique')
#generate a binary variable that indicates if there are multiple conflicts in the same country
pre_treatment['multiple_conflicts_binary'] = (pre_treatment['multiple_conflicts'] > 1).astype(int)

#number of factions involved in the conflict
pre_treatment['n_factions_combined'] = pre_treatment['n_factions_combined'].fillna(1)
pre_treatment['has_multiple_factions'] = (pre_treatment['n_factions_combined'] > 1).astype(int)


# ── 5. IA TIME-VARYING: learning inside conflicts ────────────────────
delta_monthly = 0.95 ** (1/12)

results = []
for cid, group in pre_treatment.groupby('conflict_id'):
    learning = 0.0
    values = []
    for _, row in group.iterrows():
        values.append(learning)
        learning = learning * delta_monthly + np.log1p(row['n_events'])
    results.extend(values)

pre_treatment['within_conflict_learning'] = results

In [26]:
pre_treatment["prior_interaction"] = (
    pre_treatment["experience_total"] > 0
).astype(int)

In [27]:
pre_treatment["log_recent_intensity"] = np.log1p(
    pre_treatment.groupby("conflict_id")["n_events"]
    .transform(lambda x: x.rolling(6, min_periods=1).mean().shift(1))
    .fillna(0)
)

pre_treatment["combat_volatility"] = (
    pre_treatment.groupby("conflict_id")["n_events"]
    .transform(lambda x: x.rolling(12, min_periods=3).std().shift(1))
    .fillna(0)
)

In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# IA Variable 1: cv_deaths — residual outcome uncertainty
# Rolling coefficient of variation of fatalities (12-month window, 1-month lag).
# High CoV = unpredictable battle outcomes = parties haven't learned relative strengths.
# Expected HR < 1: higher uncertainty → agreement further away.
# ─────────────────────────────────────────────────────────────────────────────
_roll_mean = (
    pre_treatment.groupby("conflict_id")["best"]
    .transform(lambda x: x.rolling(12, min_periods=3).mean().shift(1))
    .fillna(0)
)
_roll_std = (
    pre_treatment.groupby("conflict_id")["best"]
    .transform(lambda x: x.rolling(12, min_periods=3).std().shift(1))
    .fillna(0)
)
pre_treatment["cv_deaths"] = _roll_std / (1 + _roll_mean)

# ─────────────────────────────────────────────────────────────────────────────
# IA Variable 2: deaths_per_event — battle decisiveness (pace of IA revelation)
# Log of rolling mean of fatalities per event (6-month window, 1-month lag).
# High = decisive engagements = private information revealed quickly.
# Expected HR > 1: decisive battles → faster IA resolution → agreement sooner.
# ─────────────────────────────────────────────────────────────────────────────
pre_treatment["_dpe"] = (
    pre_treatment["best"] / pre_treatment["n_events"].fillna(0).clip(lower=1)
)
pre_treatment["deaths_per_event"] = np.log1p(
    pre_treatment.groupby("conflict_id")["_dpe"]
    .transform(lambda x: x.rolling(6, min_periods=1).mean().shift(1))
    .fillna(0)
)
pre_treatment.drop(columns=["_dpe"], inplace=True)

# ─────────────────────────────────────────────────────────────────────────────
# IA Variable 3: remaining_ia — decaying IA stock
# IA(t) = initial_IA × 1 / (1 + within_conflict_learning)
#   initial_IA  = 1 / (1 + IHS(experience_total))  → 1 if zero prior fighting experience
#   decay factor → shrinks as within-conflict fighting accumulates
# Most theoretically grounded: directly models the stock of unresolved private information.
# Expected HR < 1: more remaining IA → further from agreement.
# ─────────────────────────────────────────────────────────────────────────────
pre_treatment["experience_total_ihs"] = np.arcsinh(pre_treatment["experience_total"])
pre_treatment["initial_ia"]   = 1.0 / (1.0 + pre_treatment["experience_total_ihs"])
pre_treatment["remaining_ia"] = pre_treatment["initial_ia"] / (1.0 + pre_treatment["within_conflict_learning"])

pre_treatment[["conflict_id", "year_mo", "cv_deaths", "deaths_per_event", "remaining_ia"]].head(8)


,conflict_id,year_mo,cv_deaths,deaths_per_event,remaining_ia
0,205,1990-04,0.000000,0.000000,0.238474
1,205,1990-05,0.000000,0.000000,0.140847
2,205,1990-06,0.000000,0.405465,0.100059
3,205,1990-07,1.444444,0.959776,0.055160
4,205,1990-08,1.214426,0.939356,0.039088
5,205,1990-09,1.365780,0.809448,0.035213
6,205,1990-10,1.484175,0.712405,0.035342
7,205,1990-11,1.580555,0.712405,0.035470


In [29]:

# ── CP time-varying index (annual V-Dem, not pre5y) ───────────────────────────
# Contemporaneous institutional quality captures how CP evolves during the conflict.
# Fearon's key prediction: CP does NOT resolve through fighting — institutional
# weakness persists or worsens, unlike IA which fighting gradually resolves.
# All variables inverted so that high score = more severe commitment problem.
# ─────────────────────────────────────────────────────────────────────────────

pre_treatment["cp_state"]          = -pre_treatment["vdem_state_territorial_control"]
pre_treatment["cp_accountability"] = -pre_treatment["vdem_horizontal_accountability"]
pre_treatment["cp_judicial"]       = -pre_treatment["vdem_judicial_constraints"]
pre_treatment["cp_legislative"]     = -pre_treatment["vdem_legislative_constraints"]
pre_treatment['cp_political_stability'] = -pre_treatment["vdem_wb_political_stability"]
#pre_treatment["cp_neopatrimonial"] =  pre_treatment["vdem_neopatrimonial"]

CP_TV_VARS = ["cp_state", "cp_accountability", "cp_judicial", "cp_legislative", "cp_political_stability"] #, "cp_neopatrimonial"]
pre_treatment[CP_TV_VARS] = pre_treatment[CP_TV_VARS].fillna(pre_treatment[CP_TV_VARS].mean())

_scaler_cp = StandardScaler()
pre_treatment["commit_index_tv"] = _scaler_cp.fit_transform(pre_treatment[CP_TV_VARS]).mean(axis=1)

print("commit_index_tv summary:")
print(pre_treatment["commit_index_tv"].describe().round(3))
print(f"\nMean within-conflict std (annual variation): "
      f"{pre_treatment.groupby('conflict_id')['commit_index_tv'].std().mean():.3f}")


commit_index_tv summary:
count    19123.000
mean        -0.000
std          0.721
min         -2.168
25%         -0.501
50%          0.101
75%          0.594
max          1.870
Name: commit_index_tv, dtype: float64

Mean within-conflict std (annual variation): 0.121


## **Quarterly Aggregation**

---

The monthly spell is collapsed to **quarters** (`spell_q`), the unit of analysis for the discrete-time ClogLog model. Time-varying covariates (fatalities, event counts, rolling IA measures, CP index) are aggregated within each conflict-quarter; time-invariant covariates (experience, initial IA) are carried forward from the first observation. Quarterly IA and CP variables are also recomputed at this frequency.

In [30]:
# ── Add year and quarter to pre_treatment ────────────────────────────────────
pre_treatment["yq"]   = pd.to_datetime(pre_treatment["year_mo"]).dt.to_period("Q")
pre_treatment["year"] = pd.to_datetime(pre_treatment["year_mo"]).dt.year
pre_treatment['start_date_q'] = pd.to_datetime(pre_treatment['start_date']).dt.to_period("Q").dt.to_timestamp()
pre_treatment['end_date_q'] = pd.to_datetime(pre_treatment['end_date']).dt.to_period("Q").dt.to_timestamp()


# ── Aggregate to quarterly spell ──────────────────────────────────────────────
_AGG = {
    "best":                           "sum",
    'start_date_q':                    "first",
    'end_date_q':                      "first",
    "n_events":                       "sum",
    "is_first_agreement":             "max",
    "ever_agreement":                 "max",
    "conflict_age":                   "max",
    "isocode":                        "first",
    "country":                        "first",
    "region":                         "first",
    "type_of_conflict":               "first",
    "experience_total":               "first",
    "initial_ia":                     "first",
    "within_conflict_learning":       "last",
    "multiple_conflicts_binary":      "max",
    "rebel_ext_support":              "max",
    "vdem_state_territorial_control": "mean",
    "vdem_horizontal_accountability": "mean",
    "vdem_judicial_constraints":      "mean",
    "vdem_legislative_constraints":   "mean",
    "vdem_wb_political_stability":    "mean",
    "vdem_neopatrimonial":            "mean",
    "influence_veto":                 "mean",
    "influence":                      "mean",
    "influence_gdp":                  "mean",
    "influence_log_gdp":              "mean",
    "n_factions_combined":            "max",
    "new_rebel_entry":                "max",
    # V-Dem pre5y — time-varying (rolling 5y mean, updates year-by-year in spell)
    "vdem_state_territorial_control_pre5y": "mean",
    "vdem_horizontal_accountability_pre5y": "mean",
    "vdem_judicial_constraints_pre5y":      "mean",
    "vdem_legislative_constraints_pre5y":   "mean",
    "vdem_neopatrimonial_pre5y":            "mean",
    "vdem_wb_political_stability_pre5y":    "mean",
    # NOTE: _pre5y_start (time-invariant) are merged directly into spell_q below,
    # not through pre_treatment, so they are NOT listed here.
}

spell_q = (
    pre_treatment.groupby(["conflict_id", "yq"])
    .agg(_AGG)
    .reset_index()
    .sort_values(["conflict_id", "yq"])
    .reset_index(drop=True)
)

# ── Merge _pre5y_start (time-invariant) directly from df_panel ────────────────
# These are constant per conflict so they bypass the monthly→quarterly aggregation.
# This also makes the code robust to cell execution order.
_pre5y_start_lookup = (
    df_panel.drop_duplicates('conflict_id')
    [['conflict_id'] + VDEM_PRE5Y_START_VARS]
)
spell_q = spell_q.merge(_pre5y_start_lookup, on='conflict_id', how='left')

print(f"\nQuarterly spell: {len(spell_q):,} rows, {spell_q['conflict_id'].nunique()} conflicts")
print(f"Events (is_first_agreement==1 at quarter): {spell_q['is_first_agreement'].sum()}")
print(f"Zero-death quarters: {(spell_q['best']==0).mean():.1%}")
print(f"\npre5y_start NaN in spell_q:")
print(spell_q[VDEM_PRE5Y_START_VARS].isna().sum())


Quarterly spell: 6,491 rows, 201 conflicts
Events (is_first_agreement==1 at quarter): 103
Zero-death quarters: 44.4%

pre5y_start NaN in spell_q:
vdem_state_territorial_control_pre5y_start      0
vdem_horizontal_accountability_pre5y_start      0
vdem_judicial_constraints_pre5y_start           0
vdem_legislative_constraints_pre5y_start        0
vdem_neopatrimonial_pre5y_start                 0
vdem_wb_political_stability_pre5y_start       118
dtype: int64


In [31]:
# ── Quarterly IA variables ────────────────────────────────────────────────────

# Quarterly conflict age and log
spell_q["conflict_age_q"]   = spell_q.groupby("conflict_id").cumcount() + 1
spell_q["log_conflict_age_q"] = np.log1p(spell_q["conflict_age_q"])

# Cumulative deaths (quarterly)
spell_q["log_cum_deaths_q"] = np.log1p(
    spell_q.groupby("conflict_id")["best"]
    .transform(lambda x: x.cumsum().shift(1).fillna(0))
)

# CoV of quarterly deaths (4-quarter rolling)
spell_q["cv_deaths_q"] = (
    spell_q.groupby("conflict_id")["best"]
    .transform(lambda x: (
        x.rolling(4, min_periods=2).std().shift(1) /
        (1 + x.rolling(4, min_periods=2).mean().shift(1))
    ).fillna(0))
)

# IHS of quarter-over-quarter change in deaths — captures intensity shifts
spell_q["delta_deaths_ihs_q"] = np.arcsinh(
    spell_q.groupby("conflict_id")["best"]
    .transform(lambda x: x.diff().shift(1).fillna(0))
)

# Deaths per event (quarterly, 4-quarter rolling)
spell_q["_dpe_q"] = spell_q["best"] / spell_q["n_events"].fillna(0).clip(lower=1)
spell_q["deaths_per_event_q"] = np.log1p(
    spell_q.groupby("conflict_id")["_dpe_q"]
    .transform(lambda x: x.rolling(4, min_periods=1).mean().shift(1).fillna(0))
)
spell_q.drop(columns=["_dpe_q"], inplace=True)

# ── CP index (quarterly) ──────────────────────────────────────────────────────
spell_q["cp_state_q"]          = -spell_q["vdem_state_territorial_control"]
spell_q["cp_accountability_q"] = -spell_q["vdem_horizontal_accountability"]
spell_q["cp_judicial_q"]       = -spell_q["vdem_judicial_constraints"]
spell_q["cp_neopatrimonial_q"] =  spell_q["vdem_neopatrimonial"]
_CP_Q = ["cp_state_q","cp_accountability_q","cp_judicial_q","cp_neopatrimonial_q"]
spell_q[_CP_Q] = spell_q[_CP_Q].fillna(spell_q[_CP_Q].mean())
_scaler_q = StandardScaler()
spell_q["commit_index_tv_q"] = _scaler_q.fit_transform(spell_q[_CP_Q]).mean(axis=1)

# ── Interaction: key Fearon test ─────────────────────────────────────────────
spell_q["logcum_x_ia"] = spell_q["log_cum_deaths_q"] * spell_q["initial_ia"]

# ── International attention variables (IHS for right-skew) ───────────────────
spell_q["influence_veto_ihs"] = np.arcsinh(spell_q["influence_veto"].fillna(0))

# ── IA binary: no prior fighting experience ───────────────────────────────────
spell_q["high_ia_bin"] = (spell_q["experience_total"] == 0).astype(int)

print("Quarterly variable summary:")
_summary_vars = ["cv_deaths_q", "delta_deaths_ihs_q", "deaths_per_event_q",
                 "commit_index_tv_q", "influence_veto_ihs", "logcum_x_ia"]
print(spell_q[_summary_vars].describe().round(3))
print(f"\nZero-death quarters: {(spell_q['best']==0).mean():.1%}  (was 82% monthly)")

Quarterly variable summary:
       cv_deaths_q  delta_deaths_ihs_q  deaths_per_event_q  commit_index_tv_q  \
count     6491.000            6491.000            6491.000           6491.000   
mean         0.529              -0.006               1.048              0.000   
std          0.533               2.944               1.043              0.831   
min          0.000             -10.998               0.000             -1.942   
25%          0.000              -1.444               0.000             -0.519   
50%          0.433               0.000               0.954              0.113   
75%          0.869               1.444               1.642              0.604   
max          2.000              12.321               7.123              1.911   

       influence_veto_ihs  logcum_x_ia  
count            6491.000     6491.000  
mean                4.030        3.008  
std                 0.778        2.093  
min                 0.000        0.000  
25%                 4.157        1.47

In [32]:
_export_cols = [
    'conflict_id', 'yq', 'start_date_q', 'end_date_q',
    # outcome / censoring
    'is_first_agreement', 'ever_agreement',
    # baseline hazard
    'conflict_age_q', 'log_conflict_age_q',
    # raw intensity
    'best', 'n_events',
    # controls — time-invariant
    'experience_total', 'initial_ia', 'high_ia_bin',
    # controls — time-varying
    'log_cum_deaths_q', 'delta_deaths_ihs_q',
    'cv_deaths_q', 'deaths_per_event_q',
    'within_conflict_learning', 'logcum_x_ia',
    'multiple_conflicts_binary', 'n_factions_combined', 'new_rebel_entry',
    'rebel_ext_support',
    'commit_index_tv_q',
    'influence_veto', 'influence_veto_ihs', 'influence_gdp', 'influence_log_gdp',
    # V-Dem pre5y — time-invariant (frozen at conflict start year)
    'vdem_state_territorial_control_pre5y_start',
    'vdem_horizontal_accountability_pre5y_start',
    'vdem_judicial_constraints_pre5y_start',
    'vdem_legislative_constraints_pre5y_start',
    'vdem_neopatrimonial_pre5y_start',
    'vdem_wb_political_stability_pre5y_start',
    # V-Dem pre5y — time-varying (rolling 5y mean evolves during spell)
    'vdem_state_territorial_control_pre5y',
    'vdem_horizontal_accountability_pre5y',
    'vdem_judicial_constraints_pre5y',
    'vdem_legislative_constraints_pre5y',
    'vdem_neopatrimonial_pre5y',
    'vdem_wb_political_stability_pre5y',
    # identifiers
    'conflict_id', 'country', 'region', 'isocode', 'type_of_conflict',
]
# deduplicate in case conflict_id appears twice
_export_cols = list(dict.fromkeys(_export_cols))

# ── CSV (for survival_analysis.ipynb) ─────────────────────────────────────────
spell_q[_export_cols].to_csv(
    '../../data/output/conflict_level/spell_q.csv', index=False
)

# ── Stata (for cloglog_hazard.do) ─────────────────────────────────────────────
# version=119 (Stata 15+) is required because several _pre5y_start variable names
# exceed 32 characters — the limit in version 118 (Stata 14).
spell_q['yq_dt'] = spell_q['yq'].dt.to_timestamp()

spell_dta = spell_q[_export_cols + ['yq_dt']].copy()
spell_dta = spell_dta.drop(columns=['yq'])

# spell_dta.to_stata(
#     '../../data/output/conflict_level/spell_q.dta',
#     convert_dates={'yq_dt': 'tq'},
#     write_index=False,
#     version=119,
# )

n, k = spell_dta.shape
print(f"Saved {n:,} rows × {k} cols")
print(f"  → ../../data/output/conflict_level/spell_q.csv")
print(f"  → ../../data/output/conflict_level/spell_q.dta")

Saved 6,491 rows × 44 cols
  → ../../data/output/conflict_level/spell_q.csv
  → ../../data/output/conflict_level/spell_q.dta


In [33]:
# Export conflict-level proxies for the Stata HTE CSDID analysis.
# Variable names are shortened to fit Stata's 32-character limit;
# the full names in spell_q.csv would cause import errors in Stata.
_proxy_rename = {
    'vdem_state_territorial_control_pre5y_start': 'vdem_state_ctrl_pre5y',
    'vdem_horizontal_accountability_pre5y_start':  'vdem_horacc_pre5y',
    'vdem_judicial_constraints_pre5y_start':       'vdem_jucon_pre5y',
    'vdem_legislative_constraints_pre5y_start':    'vdem_legcon_pre5y',
    'vdem_neopatrimonial_pre5y_start':             'vdem_neopat_pre5y',
    'vdem_wb_political_stability_pre5y_start':     'vdem_polstab_pre5y',
}
conflict_proxies = (
    spell_q
    .drop_duplicates('conflict_id')
    [['conflict_id', 'experience_total', 'initial_ia', 'high_ia_bin'] +
     list(_proxy_rename.keys())]
    .rename(columns=_proxy_rename)
)
conflict_proxies.to_csv(
    '../../data/output/conflict_level/conflict_proxies_extended.csv', index=False
)
print(f"Saved conflict_proxies.csv: {len(conflict_proxies)} conflicts x {conflict_proxies.shape[1]} cols")
print(conflict_proxies[['experience_total', 'initial_ia', 'high_ia_bin']].describe().round(3))

Saved conflict_proxies.csv: 201 conflicts x 10 cols
       experience_total  initial_ia  high_ia_bin
count           201.000     201.000      201.000
mean              3.451       0.627        0.383
std               4.475       0.330        0.487
min               0.000       0.217        0.000
25%               0.000       0.277        0.000
50%               0.880       0.557        0.000
75%               6.732       1.000        1.000
max              18.576       1.000        1.000


## **Export**

---

Save the quarterly spell dataset for use in `survival_analysis.ipynb`.
- `spell_q.csv` — loaded by the analysis notebook (Python)
- `spell_q.dta` — for the Stata ClogLog specification (`cloglog_hazard.do`)